# RetainAI — Block 2: Production on SageMaker

1. Shared feature code with a training/serving parity test
2. Bayesian hyperparameter tuning as managed SageMaker training jobs
3. Serverless endpoint (pay per request, $0 when idle)
4. Latency benchmark
5. Lambda API that accepts a raw customer record and returns churn risk

In [1]:
%pip install -q "sagemaker>=2.200,<3"

Note: you may need to restart the kernel to use updated packages.


## 1. Setup and shared features

In [2]:
import os, sys, json, time
import numpy as np, pandas as pd, sagemaker
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

sys.path.insert(0, "src")
from features import build_frame, encode_record, RAW_FIELDS

session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = session.default_bucket()
region = session.boto_region_name
prefix = "retainai"
ENDPOINT_NAME = "retainai-churn-serverless"

raw = pd.read_csv("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
X, y = build_frame(raw)
COLUMNS = list(X.columns)
print("Region:", region, "| Bucket:", bucket)
print("Customers:", len(X), "| Features:", len(COLUMNS))

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Region: us-east-1 | Bucket: sagemaker-us-east-1-688060218999
Customers: 7043 | Features: 34


## 2. Training/serving parity test

The #1 silent bug in production ML is *training-serving skew*: features computed one way in training
and slightly differently at inference. This checks that the pure-Python encoder used in Lambda produces
exactly the same vectors as the pandas pipeline used for training.

In [3]:
sample = raw.sample(500, random_state=0)
mismatches = 0
for idx, rec in sample.iterrows():
    online = np.array(encode_record(rec.to_dict(), COLUMNS))
    if not np.allclose(online, X.loc[idx].values):
        mismatches += 1
print(f"Parity check: {500 - mismatches}/500 customers identical")
assert mismatches == 0, "Training/serving skew detected!"

Parity check: 500/500 customers identical


## 3. Split and upload to S3 (XGBoost format: label first, no header)

In [4]:
os.makedirs("data", exist_ok=True)
data = pd.concat([y.rename("Churn"), X], axis=1)
train, temp = train_test_split(data, test_size=0.3, stratify=data["Churn"], random_state=42)
val, test = train_test_split(temp, test_size=0.5, stratify=temp["Churn"], random_state=42)

train.to_csv("data/train.csv", header=False, index=False)
val.to_csv("data/validation.csv", header=False, index=False)
train_s3 = session.upload_data("data/train.csv", bucket, f"{prefix}/train")
val_s3 = session.upload_data("data/validation.csv", bucket, f"{prefix}/validation")
print(train_s3, val_s3, sep="\n")

s3://sagemaker-us-east-1-688060218999/retainai/train/train.csv
s3://sagemaker-us-east-1-688060218999/retainai/validation/validation.csv


## 4. Bayesian hyperparameter tuning (managed training jobs)

8 training jobs, 2 at a time. SageMaker's Bayesian search uses results from finished jobs to choose
the next hyperparameters. Expect ~15–20 minutes. Watch it live in Studio under **Jobs → Training**.

In [5]:
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.tuner import HyperparameterTuner, IntegerParameter, ContinuousParameter

image = sagemaker.image_uris.retrieve("xgboost", region, version="1.7-1")
xgb = Estimator(
    image_uri=image, role=role, instance_count=1, instance_type="ml.m5.large",
    output_path=f"s3://{bucket}/{prefix}/output", sagemaker_session=session,
    base_job_name="retainai-xgb",
)
xgb.set_hyperparameters(objective="binary:logistic", eval_metric="auc",
                        num_round=400, early_stopping_rounds=25)

ranges = {
    "eta": ContinuousParameter(0.01, 0.3, scaling_type="Logarithmic"),
    "max_depth": IntegerParameter(2, 6),
    "min_child_weight": ContinuousParameter(1, 10),
    "subsample": ContinuousParameter(0.6, 1.0),
    "colsample_bytree": ContinuousParameter(0.5, 1.0),
    "lambda": ContinuousParameter(0.1, 10, scaling_type="Logarithmic"),
}

tuner = HyperparameterTuner(
    xgb, objective_metric_name="validation:auc", objective_type="Maximize",
    hyperparameter_ranges=ranges, strategy="Bayesian",
    max_jobs=8, max_parallel_jobs=2, early_stopping_type="Auto",
    base_tuning_job_name="retainai",
)
tuner.fit({
    "train": TrainingInput(train_s3, content_type="text/csv"),
    "validation": TrainingInput(val_s3, content_type="text/csv"),
})

/opt/conda/lib/python3.12/site-packages/sagemaker/estimator.py:588: SageMakerV2DeprecationWarning: Estimator is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
Estimator is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
/opt/conda/lib/python3.12/site-packages/sagemaker/tuner.py:689: SageMakerV2DeprecationWarning: HyperparameterTuner is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use the hyperparameter tuning classes under sagemaker.c

.............................................................................!


In [6]:
results = tuner.analytics().dataframe().sort_values("FinalObjectiveValue", ascending=False)
results[["TrainingJobName", "FinalObjectiveValue", "eta", "max_depth", "min_child_weight",
         "subsample", "colsample_bytree", "lambda"]].head(8).round(4)

,TrainingJobName,FinalObjectiveValue,eta,max_depth,min_child_weight,subsample,colsample_bytree,lambda
0,retainai-261002-0312-008-2e18dceb,0.8528,0.0363,4.0,8.5034,0.7315,0.7756,2.7557
1,retainai-261002-0312-007-91db5e2f,0.8521,0.0263,4.0,3.1049,0.6535,0.5061,0.3213
4,retainai-261002-0312-004-474e2b48,0.8521,0.0371,4.0,6.7132,0.8241,0.7713,7.0602
5,retainai-261002-0312-003-04983681,0.8512,0.0145,2.0,3.4280,0.6262,0.7095,1.6171
6,retainai-261002-0312-002-35a1e229,0.8512,0.0203,6.0,3.3354,0.9139,0.9190,1.3784
3,retainai-261002-0312-005-2f840d6a,0.8511,0.0568,2.0,5.3211,0.6578,0.6568,0.1197
2,retainai-261002-0312-006-4a284ba6,0.8508,0.1158,3.0,5.8799,0.7945,0.6671,0.2801
7,retainai-261002-0312-001-3369ff38,0.8488,0.1715,5.0,6.9255,0.9079,0.8748,2.9873


## 5. Deploy the best model to a serverless endpoint

In [7]:
from sagemaker.serverless import ServerlessInferenceConfig
from sagemaker.serializers import CSVSerializer

best = tuner.best_estimator()
predictor = best.deploy(
    serverless_inference_config=ServerlessInferenceConfig(memory_size_in_mb=2048, max_concurrency=2),
    serializer=CSVSerializer(),
    endpoint_name=ENDPOINT_NAME,
)
print("Deployed:", predictor.endpoint_name)


2026-10-02 03:18:45 Starting - Found matching resource for reuse
2026-10-02 03:18:45 Downloading - Downloading the training image
2026-10-02 03:18:45 Training - Training image download completed. Training in progress.
2026-10-02 03:18:45 Uploading - Uploading generated training model
2026-10-02 03:18:45 Completed - Resource retained for reuse

/opt/conda/lib/python3.12/site-packages/sagemaker/model.py:347: SageMakerV2DeprecationWarning: Model is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
Model is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.



-----!

/opt/conda/lib/python3.12/site-packages/sagemaker/base_predictor.py:140: SageMakerV2DeprecationWarning: Predictor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.resources.Endpoint`.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
Predictor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.resources.Endpoint`.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Deployed: retainai-churn-serverless


## 6. Evaluate on held-out test set + latency benchmark

In [8]:
def predict(rows):
    out = predictor.predict(rows).decode("utf-8")
    return [float(p) for p in out.replace("\n", ",").split(",") if p.strip()]

X_test, y_test = test.drop(columns=["Churn"]).values, test["Churn"].values
preds = []
for batch in np.array_split(X_test, 10):
    preds += predict(batch)
print("Test AUC (served model):", round(roc_auc_score(y_test, preds), 4))

# Latency: first call may include a cold start; then measure warm calls
t0 = time.time(); predict(X_test[:1]); cold = (time.time() - t0) * 1000
times = []
for i in range(20):
    t0 = time.time(); predict(X_test[i:i+1]); times.append((time.time() - t0) * 1000)
print(f"First call: {cold:.0f} ms | warm p50: {np.percentile(times, 50):.0f} ms | warm p95: {np.percentile(times, 95):.0f} ms")

Test AUC (served model): 0.8425
First call: 39 ms | warm p50: 41 ms | warm p95: 45 ms


## 7. Build the Lambda API

Assembles one deployable file: shared feature code + the column list + the handler.
Then prints a real customer record to use as a test event.

In [9]:
os.makedirs("api", exist_ok=True)
lambda_src = (
    open("src/features.py").read()
    + "\n\nCOLUMNS = " + json.dumps(COLUMNS) + "\n\n"
    + open("api/handler.py").read()
)
with open("api/lambda_function.py", "w") as f:
    f.write(lambda_src)
compile(lambda_src, "lambda_function.py", "exec")   # syntax check
print("Wrote api/lambda_function.py", f"({len(lambda_src.splitlines())} lines)")

os.makedirs("artifacts", exist_ok=True)
json.dump({"endpoint": ENDPOINT_NAME, "columns": COLUMNS}, open("artifacts/deployment.json", "w"))

high_risk = raw.loc[test.index].assign(p=preds).sort_values("p", ascending=False).iloc[0]
event = {k: (v.item() if hasattr(v, "item") else v) for k, v in high_risk[["customerID"] + RAW_FIELDS].items()}
print("\nTest event for the Lambda console:\n")
print(json.dumps(event, indent=2))

Wrote api/lambda_function.py (113 lines)

Test event for the Lambda console:

{
  "customerID": "5178-LMXOP",
  "SeniorCitizen": 1,
  "tenure": 1,
  "MonthlyCharges": 95.1,
  "TotalCharges": "95.1",
  "gender": "Male",
  "Partner": "Yes",
  "Dependents": "No",
  "PhoneService": "Yes",
  "MultipleLines": "Yes",
  "InternetService": "Fiber optic",
  "OnlineSecurity": "No",
  "OnlineBackup": "No",
  "DeviceProtection": "No",
  "TechSupport": "No",
  "StreamingTV": "Yes",
  "StreamingMovies": "Yes",
  "Contract": "Month-to-month",
  "PaperlessBilling": "Yes",
  "PaymentMethod": "Electronic check"
}


In [ ]:
git add .
git commit -m "Block 2: shared features, Bayesian tuning, serverless endpoint, Lambda API"
git push origin main